In [ ]:
import pandas as pd
import numpy as np

# ============================
# PARAMETERS
# ============================

SIMULATIONS = 1000
SHORTAGE_PENALTY = 10
HOLDING_PENALTY = 1
AVAILABLE_HOURS = 22

# ============================
# LOAD FILES
# ============================

stats = pd.read_excel("stats_file.xlsx")
daily = pd.read_excel("daily_demand.xlsx")

data = stats.merge(daily, on="Material")

# ============================
# PRODUCTION RATE
# ============================

data["Rate"] = (3600 / data["Cycle_Time"]) * data["Cavity"]

# ============================
# OPTIMIZATION FUNCTION
# ============================

def optimize_part(row):

    inventory = row["Inventory"]
    demand_tomorrow = row["Actual_Tomorrow"]
    tentative_future = row["Tentative_Future"]
    std = row["Std_Deviation"]
    rate = row["Rate"]

    best_qty = 0
    best_cost = np.inf

    for qty in np.arange(0, 5000, 50):

        future_stock = inventory + qty - demand_tomorrow

        simulated_demand = np.random.normal(
            tentative_future, std, SIMULATIONS
        )

        shortage = np.maximum(0, simulated_demand - future_stock)

        expected_shortage = shortage.mean()
        expected_inventory = max(0, future_stock - simulated_demand.mean())

        cost = (
            SHORTAGE_PENALTY * expected_shortage +
            HOLDING_PENALTY * expected_inventory
        )

        hours_needed = qty / rate

        if hours_needed <= AVAILABLE_HOURS and cost < best_cost:
            best_cost = cost
            best_qty = qty

    return best_qty, best_cost

# ============================
# RUN OPTIMIZATION
# ============================

results = data.apply(optimize_part, axis=1)

data["Planned_Qty"] = [r[0] for r in results]
data["Cost"] = [r[1] for r in results]

data.to_excel("daily_plan.xlsx", index=False)

print("✅ Optimization complete")